In [1]:
import pandas as pd
import nltk
from nltk.tokenize import sent_tokenize

In [2]:
def download_nltk_data():
    try:
        nltk.data.find('tokenizers/punkt')
    except LookupError:
        print("Downloading NLTK punkt...")
        nltk.download('punkt', quiet=True)
        print("   Done")


def ease_augment(df, text_col='text', label_col='priority', min_length=15):    
    download_nltk_data()
    
    print("="*80)
    print("EASE AUGMENTATION")
    print("="*80)
    print(f"Original samples: {len(df)}")
    print(f"Min sentence length: {min_length} characters\n")
    
    augmented_samples = []
    total_extracted = 0
    total_kept = 0
    
    for idx, row in df.iterrows():
        text = str(row[text_col])
        label = row[label_col]
        
        # Step 1: Extract sentences using NLTK
        sentences = sent_tokenize(text)
        total_extracted += len(sentences)
        
        # Skip if only 1 sentence (no augmentation needed)
        if len(sentences) <= 1:
            continue
        
        # Step 2: Acquire labels (use original label due to unavailabel pre-trained models for bug serverity level prediction)
        
        # Step 3: Sift (filter by length)
        for sentence in sentences:
            sentence = sentence.strip()
            
            # Filter short sentences
            if len(sentence) >= min_length:
                augmented_samples.append({
                    text_col: sentence,
                    label_col: label
                })
                total_kept += 1
        
        # Progress
        if (idx + 1) % 100 == 0:
            print(f"   Processed: {idx+1}/{len(df)}", end='\r')
    
    print(f"   Processed: {len(df)}/{len(df)}")
    
    # Step 4: Employ (merge with original)
    df_aug = pd.DataFrame(augmented_samples)
    df_combined = pd.concat([df, df_aug], ignore_index=True)
    
    # Statistics
    augmentation_rate = len(df_aug) / len(df) if len(df) > 0 else 0
    
    print(f"\n AUGMENTATION COMPLETED")
    print("="*80)
    print(f"Original samples:      {len(df):6d}")
    print(f"Augmented samples:     {len(df_aug):6d}")
    print(f"Total samples:         {len(df_combined):6d}")
    print(f"Augmentation rate:     {augmentation_rate:.2f}x")
    print(f"\nSentences extracted:   {total_extracted:6d}")
    print(f"Sentences kept:        {total_kept:6d}")
    print(f"Keep rate:             {total_kept/total_extracted*100:.1f}%")
    
    print(f"\n Label Distribution:")
    print("="*80)
    print("ORIGINAL:")
    print(df[label_col].value_counts().sort_index())
    print("\nAUGMENTED:")
    print(df_aug[label_col].value_counts().sort_index())
    print("\nCOMBINED:")
    print(df_combined[label_col].value_counts().sort_index())
    print("="*80)
    
    return df_combined

In [3]:
df_train = pd.read_csv(r"../../datasets/raw_datasets_processed/train.csv")
df_train = df_train[["text", "priority"]]
df_train

,text,priority
0,Detect uncaught exceptions in UoWProcessor to ...,Major
1,MinaConsumerTest failure The test fails with t...,Major
2,cleanup dialog for unable to delete host make ...,Major
3,camel-test-blueprint - Allow to register custo...,Major
4,Automatic Image Linking broken for images in b...,Major
...,...,...
3193,Smpp component schedule delivery time should d...,Major
3194,NullPointerException or ClassCastException if ...,Trivial
3195,EventHelper.notifyRouteStarted skips all remai...,Minor
3196,Identity column can be created with wrong and...,Major


In [4]:
df_train = ease_augment(df_train, min_length=15)
df_train

EASE AUGMENTATION
Original samples: 3198
Min sentence length: 15 characters

   Processed: 3198/3198

 AUGMENTATION COMPLETED
Original samples:        3198
Augmented samples:       7215
Total samples:          10413
Augmentation rate:     2.26x

Sentences extracted:    11244
Sentences kept:          7215
Keep rate:             64.2%

 Label Distribution:
ORIGINAL:
priority
Blocker       39
Critical     132
Major       2078
Minor        802
Trivial      147
Name: count, dtype: int64

AUGMENTED:
priority
Blocker       95
Critical     353
Major       4675
Minor       1849
Trivial      243
Name: count, dtype: int64

COMBINED:
priority
Blocker      134
Critical     485
Major       6753
Minor       2651
Trivial      390
Name: count, dtype: int64


,text,priority
0,Detect uncaught exceptions in UoWProcessor to ...,Major
1,MinaConsumerTest failure The test fails with t...,Major
2,cleanup dialog for unable to delete host make ...,Major
3,camel-test-blueprint - Allow to register custo...,Major
4,Automatic Image Linking broken for images in b...,Major
...,...,...
10408,int numBadStartWith = 0; try { // create 1000...,Major
10409,Break out if we get a table that has a bad // ...,Major
10410,for (int i = 0; (i &lt; 1000) &amp;&amp; (numB...,Major
10411,Sending to http endpoint may double encoding p...,Major


In [5]:
df_train.to_csv(r"../../datasets/ease_aug/train_ease.csv", index=None, encoding="utf-8-sig")